# DSN AI Bootcamp 2026 Qualification Hackathon: Retail Sales Prediction

**Author**: Abdulroqeeb  
**Track**: Machine Learning  
**Problem**: Predict store-product total sales across retail outlets in Nigeria  
**Evaluation Metric**: Root Mean Squared Error (RMSE)  

---
### Notebook Structure
1. **Environment Setup & Imports**
2. **Data Loading & Structure Inspection**
3. **Exploratory Data Analysis (EDA)**
4. **Feature Engineering & Preprocessing Pipeline**
5. **Validation Strategy (Stratified by Distribution & Multi-Seed CV)**
6. **Modeling: Direct Sales + Units Demand Ensembling (CatBoost)**
7. **Blending Optimization & Diagnostics**
8. **Submission Generation**

## 1. Environment Setup & Imports
Importing standard data analysis, visualisation, and modeling libraries.

In [2]:
!pip install catboost

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.8/100.2 MB 2.4 MB/s eta 0:00:42
    --------------------------------------- 1.3/100.2 MB 2.3 MB/s eta 0:00:43
    --------------------------------------- 2.4/100.2 MB 3.2 MB/s eta 0:00:31
   - -------------------------------------- 3.1/100.2 MB 3.5 MB/s eta 0:00:28
   - -------------------------------------- 3.7/100.2 MB 3.4 MB/s eta 0:00:29
   - -------------------------------------- 4.5/100.2 MB 3.3 MB/s eta 0:00:29
   - -------------------------------------- 5.0/100.2 MB 3.2 MB/s eta 0:00:30
   -- ------------------------------------- 5.2/100.2 MB 3.2 MB/s eta 0:00:30
   -- -----------------------

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
import catboost as cb

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries successfully imported.')

ModuleNotFoundError: No module named 'catboost'

## 2. Data Loading & Structure Inspection

In [ ]:
# Load train and test sets
train_path = '../data/train.csv' if os.path.exists('../data/train.csv') else 'data/train.csv'
test_path  = '../data/test.csv' if os.path.exists('../data/test.csv') else 'data/test.csv'

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head()

In [ ]:
train_df.info()

print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

print(test_df.isnull().sum()[test_df.isnull().sum() > 0])

## 3. Exploratory Data Analysis (EDA)

Understanding the target variable distribution and relationships between outlets, prices, and visibility.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(train_df['total_sales'], kde=True, bins=40, color='#1f77b4', ax=axes[0])
axes[0].set_title('Target Distribution: Total Sales (Right-Skewed)')
axes[0].set_xlabel('Total Sales (NGN)')

sns.histplot(np.log1p(train_df['total_sales']), kde=True, bins=40, color='#2ca02c', ax=axes[1])
axes[1].set_title('Distribution of log1p(Total Sales)')
axes[1].set_xlabel('log1p(Total Sales)')

In [ ]:
# Store volume and format analysis
plt.figure(figsize=(12, 5))
store_order = train_df.groupby('store_code')['total_sales'].mean().sort_values(ascending=False).index
sns.barplot(data=train_df, x='store_code', y='total_sales', order=store_order, palette='Blues_r', ci=None)
plt.title('Average Sales by Store Outlet')
plt.xticks(rotation=45)
plt.ylabel('Mean Total Sales')

In [ ]:
# Price vs Sales by Store Format
plt.figure(figsize=(10, 6))
sns.scatterplot(data=train_df.sample(1500, random_state=42), x='product_price', y='total_sales', hue='store_format', alpha=0.6)
plt.title('Product Price vs. Total Sales by Store Format')

### Takeaways & Insights
1. **Disjoint Assortment**: Test records do not share product-store combinations with train records. The model must learn intrinsic category and format behavior rather than memorising specific product-outlet pairs.
2. **Zero Shelf Visibility**: Approximately 6% of records have shelf_visibility = 0.0. In actual retail, products actively sold cannot occupy 0% display area. These represent missing values masked as zeros.
3. **Store Tiering**: Clear clustering between high-capacity hypermarkets (STORE-7WS), standard supermarkets, and corner stores (STORE-JOR, STORE-T5G).
4. **Price Collinearity**: Total sales strongly correlate with product_price. Decomposing sales into units_sold = total_sales / product_price yields a stationary demand target.

## 4. Feature Engineering & Preprocessing Pipeline

We construct a leakage-free preprocessing pipeline:
- Category spelling and casing standardization
- Imputation of product_weight_kg and store_size using hierarchical lookups
- Zero-visibility replacement via item-level and category-level medians
- Normalized price benchmarks and store format capacity multipliers
- Relative interactions between product category and store format

In [ ]:
def build_features(train, test):
    n_tr = len(train)
    y = train['total_sales'].copy()
    df = pd.concat([train.drop(columns=['total_sales']), test], ignore_index=True)

    # Clean Category and Fat Content
    df['product_category'] = df['product_category'].str.strip().str.title()
    non_edible = ['Health And Hygiene', 'Household', 'Others']
    df['is_edible'] = (~df['product_category'].isin(non_edible)).astype(int)
    df.loc[df['is_edible'] == 0, 'fat_content'] = 'Non-Edible'

    # Impute Product Weights
    weight_map = df.dropna(subset=['product_weight_kg']).groupby('product_code')['product_weight_kg'].mean()
    df['product_weight_kg'] = df['product_weight_kg'].fillna(df['product_code'].map(weight_map))
    cat_w = df.groupby('product_category')['product_weight_kg'].transform('median')
    df['product_weight_kg'] = df['product_weight_kg'].fillna(cat_w).fillna(df['product_weight_kg'].median())

    # Impute Store Size
    size_map = {'STORE-JOR': 'Small', 'STORE-DKU': 'Small', 'STORE-OYG': 'Small'}
    df['store_size'] = df['store_size'].fillna(df['store_code'].map(size_map))

    # Correct Zero Visibility
    df['visibility_was_zero'] = (df['shelf_visibility'] == 0.0).astype(int)
    pos_vis = df[df['shelf_visibility'] > 0]
    p_vis = pos_vis.groupby('product_code')['shelf_visibility'].mean()
    c_vis = pos_vis.groupby('product_category')['shelf_visibility'].median()
    zero_mask = df['shelf_visibility'] == 0.0
    df.loc[zero_mask, 'shelf_visibility'] = df.loc[zero_mask, 'product_code'].map(p_vis)
    df['shelf_visibility'] = df['shelf_visibility'].fillna(df['product_category'].map(c_vis)).fillna(df['shelf_visibility'].median())

    # Price Benchmarks & Relative Metrics
    df['log_price'] = np.log1p(df['product_price'])
    df['sqrt_price'] = np.sqrt(df['product_price'])
    for grp in ['product_category', 'store_format', 'store_location_tier']:
        g_mean = df.groupby(grp)['product_price'].transform('mean')
        g_std  = df.groupby(grp)['product_price'].transform('std').replace(0, 1)
        df[f'price_to_{grp}_mean'] = df['product_price'] / g_mean
        df[f'price_z_{grp}'] = (df['product_price'] - g_mean) / g_std

    # Ordinal Encodings & Volume Multipliers
    fmt_ord  = {'Corner Shop': 1, 'Superstore': 2, 'Standard Supermarket': 3, 'Flagship Hypermarket': 4}
    tier_ord = {'Tier 1': 3, 'Tier 2': 2, 'Tier 3': 1}
    sz_ord   = {'Small': 1, 'Medium': 2, 'Large': 3}
    df['store_format_ord'] = df['store_format'].map(fmt_ord).fillna(2)
    df['store_tier_ord']   = df['store_location_tier'].map(tier_ord).fillna(2)
    df['store_size_ord']   = df['store_size'].map(sz_ord).fillna(2)
    df['store_volume_mult'] = df['store_format_ord'] * df['store_tier_ord']

    # Interaction Terms
    df['price_x_volume'] = df['product_price'] * df['store_volume_mult']
    df['price_x_size']   = df['product_price'] * df['store_size_ord']
    df['price_x_age']    = df['product_price'] * df['store_age_years']
    df['cat_store_combo']  = df['product_category'] + '_' + df['store_code']
    df['cat_format_combo'] = df['product_category'] + '_' + df['store_format']

    # Categorical labeling for GBDTs
    cat_cols = [
        'product_code', 'fat_content', 'product_category',
        'store_code', 'store_size', 'store_location_tier', 'store_format',
        'cat_store_combo', 'cat_format_combo'
    ]
    for c in cat_cols:
        le = LabelEncoder()
        df[c] = le.fit_transform(df[c].astype(str))

    train_out = df.iloc[:n_tr].copy().reset_index(drop=True)
    test_out  = df.iloc[n_tr:].copy().reset_index(drop=True)
    train_out['total_sales'] = y.values
    return train_out, test_out, cat_cols

train_feat, test_feat, cat_features = build_features(train_df, test_df)
print(f'Feature engineering complete. Total feature columns: {train_feat.shape[1] - 2}')

## 5. Multi-Seed Cross-Validation & Modeling

### Modeling Paradigm:
- **Direct Sales Model**: Models nominal NGN sales with RMSE loss.
- **Units Demand Model**: Predicts sales volume normalized by product price, multiplied back at inference.
- **Multi-Seed Bagging**: Trained across multiple distinct random seeds to minimize fold variance.

In [ ]:
features = [c for c in train_feat.columns if c not in ['id', 'total_sales']]
X = train_feat[features]
y = train_feat['total_sales'].values
prices_tr = train_feat['product_price'].values

X_test = test_feat[features]
prices_te = test_feat['product_price'].values

seeds = [42, 2026, 777]
n_splits = 5

oof_direct = np.zeros(len(X))
oof_units  = np.zeros(len(X))
test_direct = np.zeros(len(X_test))
test_units  = np.zeros(len(X_test))

cb_cats = [c for c in cat_features if c in X.columns]

print(f'Training CatBoost ensemble across {len(seeds)} seeds...')
for seed in seeds:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]
        
        # 1. Direct Sales Regressor
        m_dir = cb.CatBoostRegressor(
            iterations=1500, learning_rate=0.03, depth=6, l2_leaf_reg=5.0,
            random_seed=seed, verbose=False
        )
        m_dir.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=cb_cats, early_stopping_rounds=40, verbose=False)
        oof_direct[val_idx] += np.clip(m_dir.predict(X_va), 0, None) / len(seeds)
        test_direct += np.clip(m_dir.predict(X_test), 0, None) / (len(seeds) * n_splits)
        
        # 2. Units Demand Regressor
        y_u_tr = y_tr / prices_tr[tr_idx]
        y_u_va = y_va / prices_tr[val_idx]
        m_u = cb.CatBoostRegressor(
            iterations=1500, learning_rate=0.03, depth=6, l2_leaf_reg=5.0,
            random_seed=seed, verbose=False
        )
        m_u.fit(X_tr, y_u_tr, eval_set=(X_va, y_u_va), cat_features=cb_cats, early_stopping_rounds=40, verbose=False)
        oof_units[val_idx] += (np.clip(m_u.predict(X_va), 0, None) * prices_tr[val_idx]) / len(seeds)
        test_units += (np.clip(m_u.predict(X_test), 0, None) * prices_te) / (len(seeds) * n_splits)

print('Model training complete.')
print(f'Direct Model OOF RMSE: {root_mean_squared_error(y, oof_direct):.2f}')
print(f'Units Model OOF RMSE:  {root_mean_squared_error(y, oof_units):.2f}')

## 6. Ensembling & Optimal Convex Blending

Finding the optimal blend weight $w$ to minimize overall out-of-fold RMSE without target leakage.

In [ ]:
best_rmse = float('inf')
best_w = 0.5
for w in np.linspace(0, 1, 101):
    blend_oof = w * oof_direct + (1 - w) * oof_units
    rmse = root_mean_squared_error(y, blend_oof)
    if rmse < best_rmse:
        best_rmse = rmse
        best_w = w

print(f'Optimal Blend: {best_w:.2f} * Direct + {1-best_w:.2f} * Units')
print(f'Ensemble Cross-Validation RMSE: {best_rmse:.2f}')

final_preds = np.clip(best_w * test_direct + (1 - best_w) * test_units, 0, None)

## 7. Submission Diagnostics & Export

In [ ]:
print('--- Test Prediction Summary ---')
print(f'Total count: {len(final_preds)}')
print(f'Prediction Mean: {final_preds.mean():.2f} (Train Mean: {y.mean():.2f})')
print(f'Prediction Std:  {final_preds.std():.2f}  (Train Std:  {y.std():.2f})')
print(f'Min Prediction:  {final_preds.min():.2f}')
print(f'Max Prediction:  {final_preds.max():.2f}')

sub_df = pd.DataFrame({
    'id': test_df['id'],
    'total_sales': np.round(final_preds, 2)
})

out_file = '../submissions/submission_notebook_pipeline.csv' if os.path.exists('../submissions') else 'submissions/submission_notebook_pipeline.csv'
os.makedirs(os.path.dirname(out_file), exist_ok=True)
sub_df.to_csv(out_file, index=False)
print(f'Submission successfully exported to {out_file}')